# Optuna: Optimización de Hiperparámetros para Machine Learning

## ¿Qué es Optuna?

**Optuna** es un framework de código abierto de optimización automática de hiperparámetros, diseñado específicamente para Machine Learning. Fue desarrollado por Preferred Networks y se ha convertido en una de las herramientas más populares y eficientes para ajuste de hiperparámetros.

### 🎯 Objetivos de este Notebook

En este tutorial aprenderás:

1. **Fundamentos de Optuna**: Qué es y por qué usarlo
2. **Conceptos Clave**: Studies, Trials, Samplers y Pruners
3. **Métodos de Búsqueda**: Grid Search vs Random Search vs Bayesian Optimization
4. **Ejemplos Prácticos**: Desde casos simples hasta complejos
5. **Visualizaciones**: Cómo analizar resultados
6. **Mejores Prácticas**: Tips para optimizar tus búsquedas

---

## ¿Por qué usar Optuna?

### Ventajas principales:

**1. 🚀 Eficiencia**
   - Algoritmos de optimización bayesiana avanzados
   >Un algoritmo bayesiano es una técnica de aprendizaje automático supervisado basada en el teorema de Bayes, que calcula la probabilidad de que ocurra un evento (hipótesis) dadas ciertas evidencias. Utiliza la probabilidad estadística para modelar la incertidumbre, siendo especialmente útil con datos limitados, ruidosos o para clasificaciones rápidas y escalables, como el filtrado de spam o diagnósticos médicos. 
   >Combina conocimientos previos (probabilidad a priori) con nuevos datos (verosimilitud o likelihood) para actualizar las creencias (probabilidad a posteriori).
   - Pruning automático de trials no prometedores
   - Paralelización nativa

**2. 🎨 Flexibilidad**
   - Define-by-run API (muy intuitiva)
   - Compatible con cualquier librería de ML (scikit-learn, XGBoost, TensorFlow, PyTorch, etc.)
   - Fácil integración en código existente

**3. 📊 Visualización**
   - Gráficos interactivos integrados
   - Análisis de importancia de hiperparámetros
   - Historial completo de optimización

**4. 💾 Persistencia**
   - Almacenamiento de estudios en bases de datos
   - Reanudación de optimizaciones interrumpidas
   - Compartición de resultados entre equipos

---

## Comparación con otras herramientas

| Característica | Optuna | Grid Search | Random Search | Hyperopt |
|----------------|--------|-------------|---------------|----------|
| Eficiencia | ⭐⭐⭐⭐⭐ | ⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| Facilidad de uso | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| Pruning automático | ✅ | ❌ | ❌ | ❌ |
| Paralelización | ✅ | ✅ | ✅ | ✅ |
| Visualizaciones | ✅ | ❌ | ❌ | ⚠️ |
| API moderna | ✅ | ✅ | ✅ | ❌ |

**¡Comencemos!**

## Instalación e Importación de Librerías

In [ ]:
# Instalación de Optuna y dependencias
# Ejecutar solo si no tienes Optuna instalado
%pip install optuna
# Para visualización avanzada (opcional)
%pip install optuna-dashboard 
# Para visualizaciones interactivas
%pip install plotly  

In [ ]:
# Importar librerías necesarias
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_diabetes, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
import warnings

# Configuración
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Librerías importadas correctamente")
print(f"📦 Versión de Optuna: {optuna.__version__}")

---

## 📚 Parte 1: Conceptos Fundamentales de Optuna

### Los 4 Conceptos Clave

#### 1. **Study (Estudio)** 📊
Un estudio es el contenedor principal que gestiona todo el proceso de optimización.
- Contiene múltiples trials (pruebas)
- Define el objetivo: minimizar o maximizar
- Almacena el historial completo de optimización

#### 2. **Trial (Prueba)** 🧪
Cada trial es una evaluación individual de un conjunto específico de hiperparámetros.
- Sugiere valores para hiperparámetros
- Ejecuta el entrenamiento del modelo
- Devuelve una métrica objetivo

#### 3. **Sampler (Muestreador)** 🎲
Define la estrategia para seleccionar los siguientes hiperparámetros a probar.
- **TPE Sampler** (Tree-structured Parzen Estimator): Por defecto, muy eficiente
- **Random Sampler**: Búsqueda aleatoria
- **Grid Sampler**: Búsqueda exhaustiva
- **CMA-ES Sampler**: Estrategia de evolución

#### 4. **Pruner (Podador)** ✂️
Detiene automáticamente trials que no son prometedores, ahorrando tiempo.
- **Median Pruner**: Detiene si está por debajo de la mediana
- **Percentile Pruner**: Basado en percentiles
- **Hyperband Pruner**: Algoritmo sofisticado de early stopping

---

### Flujo de Trabajo Típico

```
1. Definir función objetivo (qué queremos optimizar)
2. Crear un Study
3. Ejecutar optimización (n_trials)
4. Analizar resultados
5. Usar los mejores hiperparámetros
```

---

## 🎯 Parte 2: Primer Ejemplo Simple - Optimizando una Función Matemática

Antes de trabajar con modelos de ML, veamos cómo funciona Optuna con una función matemática simple.

### Objetivo: Minimizar la función f(x, y) = (x - 2)² + (y + 3)²

El mínimo de esta función está en x=2, y=-3, donde f(2, -3) = 0

In [ ]:
def funcion_objetivo_simple(trial):
    """
    Función objetivo que queremos minimizar.
    
    Optuna buscará los valores de x e y que minimicen esta función.
    """
    # Sugerimos valores para x e y
    x = trial.suggest_float('x', -10, 10)
    y = trial.suggest_float('y', -10, 10)
    
    # Calculamos el valor de la función
    resultado = (x - 2)**2 + (y + 3)**2
    
    return resultado

# Crear un estudio para minimizar
study = optuna.create_study(
    direction='minimize',  # Queremos minimizar
    study_name='Ejemplo_Simple'
)

# Ejecutar la optimización con 100 trials
print("🔍 Iniciando optimización...")
study.optimize(funcion_objetivo_simple, n_trials=100)

# Mostrar resultados
print("\n" + "="*60)
print("📊 RESULTADOS DE LA OPTIMIZACIÓN")
print("="*60)
print(f"\n✅ Mejor valor encontrado: {study.best_value:.6f}")
print(f"📌 Mejores parámetros:")
for param, value in study.best_params.items():
    print(f"   - {param}: {value:.6f}")
print(f"\n🎯 Valor óptimo teórico: x=2, y=-3, f(2,-3)=0")
print(f"📈 Total de trials completados: {len(study.trials)}")

### 📊 Análisis de Resultados

Veamos cómo evolucionó la búsqueda:

In [ ]:
# Crear visualización del historial de optimización
trials_data = []
for i, trial in enumerate(study.trials):
    trials_data.append({
        'Trial': i,
        'Valor': trial.value,
        'x': trial.params['x'],
        'y': trial.params['y']
    })

df_trials = pd.DataFrame(trials_data)

# Gráfico de convergencia
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Evolución del mejor valor
best_so_far = df_trials['Valor'].cummin()
axes[0].plot(df_trials['Trial'], df_trials['Valor'], 'o', alpha=0.3, label='Cada trial')
axes[0].plot(df_trials['Trial'], best_so_far, 'r-', linewidth=2, label='Mejor hasta ahora')
axes[0].axhline(y=0, color='g', linestyle='--', label='Óptimo teórico')
axes[0].set_xlabel('Número de Trial')
axes[0].set_ylabel('Valor de la Función')
axes[0].set_title('Convergencia de la Optimización')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Distribución de puntos explorados
scatter = axes[1].scatter(df_trials['x'], df_trials['y'], 
                          c=df_trials['Valor'], cmap='viridis', 
                          alpha=0.6, s=50)
axes[1].plot(2, -3, 'r*', markersize=20, label='Óptimo teórico')
axes[1].plot(study.best_params['x'], study.best_params['y'], 
             'g*', markersize=20, label='Mejor encontrado')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Espacio de Búsqueda Explorado')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1], label='Valor función')

plt.tight_layout()
plt.show()

print(f"\n💡 Observa cómo Optuna concentra la búsqueda alrededor del óptimo después de varias iteraciones")

---

## 🤖 Parte 3: Optimización de Modelos de Machine Learning

### Ejemplo 1: Clasificación con Random Forest

Vamos a optimizar los hiperparámetros de un Random Forest Classifier en el dataset Iris.

In [ ]:
# Cargar y preparar datos
print("📊 Cargando dataset Iris...")
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)

print(f"✅ Datos preparados:")
print(f"   - Entrenamiento: {X_train.shape[0]} muestras")
print(f"   - Test: {X_test.shape[0]} muestras")
print(f"   - Características: {X_train.shape[1]}")
print(f"   - Clases: {len(np.unique(y_train))}")

In [ ]:
def objective_random_forest(trial):
    """
    Función objetivo para optimizar Random Forest Classifier.
    
    Optuna sugerirá diferentes valores de hiperparámetros
    y evaluaremos el modelo con validación cruzada.
    """
    
    # Definir el espacio de búsqueda de hiperparámetros
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 200),
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy'])
    }
    
    # Crear y entrenar el modelo
    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    
    # Evaluación con validación cruzada
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    # Retornamos la media de los scores
    # Optuna buscará maximizar este valor
    return scores.mean()

# Crear estudio (queremos MAXIMIZAR la accuracy)
study_rf = optuna.create_study(
    direction='maximize',
    study_name='Random_Forest_Optimization',
    sampler=optuna.samplers.TPESampler(seed=42)  # Reproducibilidad
)

# Ejecutar optimización
print("🔍 Optimizando Random Forest...")
print("⏳ Esto puede tomar un momento...\n")

study_rf.optimize(objective_random_forest, n_trials=50, show_progress_bar=True)

# Resultados
print("\n" + "="*70)
print("📊 RESULTADOS DE LA OPTIMIZACIÓN - RANDOM FOREST")
print("="*70)
print(f"\n✅ Mejor accuracy (CV): {study_rf.best_value:.4f}")
print(f"\n📌 Mejores hiperparámetros:")
for param, value in study_rf.best_params.items():
    print(f"   - {param}: {value}")

### Evaluar el Mejor Modelo en el Conjunto de Test

In [ ]:
# Entrenar el mejor modelo con todo el conjunto de entrenamiento
best_rf = RandomForestClassifier(**study_rf.best_params, random_state=42)
best_rf.fit(X_train, y_train)

# Evaluar en test
y_pred = best_rf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print("="*70)
print("🎯 EVALUACIÓN EN CONJUNTO DE TEST")
print("="*70)
print(f"\n📈 Accuracy en Test: {test_accuracy:.4f}")
print(f"📈 Accuracy en CV (entrenamiento): {study_rf.best_value:.4f}")

# Comparar con modelo por defecto
default_rf = RandomForestClassifier(random_state=42)
default_rf.fit(X_train, y_train)
default_accuracy = accuracy_score(y_test, default_rf.predict(X_test))

print(f"\n🔄 Comparación:")
print(f"   - Modelo optimizado: {test_accuracy:.4f}")
print(f"   - Modelo por defecto: {default_accuracy:.4f}")
print(f"   - Mejora: {((test_accuracy - default_accuracy) / default_accuracy * 100):.2f}%")

---

## 📊 Parte 4: Visualizaciones Avanzadas de Optuna

Optuna incluye funciones de visualización muy útiles para analizar el proceso de optimización.

In [ ]:
# 1. Historial de optimización
fig = optuna.visualization.plot_optimization_history(study_rf)
fig.update_layout(
    title="Historial de Optimización - Random Forest",
    xaxis_title="Número de Trial",
    yaxis_title="Accuracy (CV)"
)
fig.show()

print("📈 Este gráfico muestra cómo mejora la accuracy con cada trial")

In [ ]:
# 2. Importancia de hiperparámetros
fig = optuna.visualization.plot_param_importances(study_rf)
fig.update_layout(
    title="Importancia de Hiperparámetros"
)
fig.show()

print("\n💡 Los hiperparámetros más arriba son los que más afectan el rendimiento")

In [ ]:
# 3. Relación entre hiperparámetros
fig = optuna.visualization.plot_slice(study_rf)
fig.update_layout(
    title="Impacto de cada Hiperparámetro en la Accuracy",
    height=800
)
fig.show()

print("\n📊 Cada panel muestra cómo un hiperparámetro afecta el objetivo")

In [ ]:
# 4. Coordenadas paralelas (relación entre múltiples parámetros)
fig = optuna.visualization.plot_parallel_coordinate(
    study_rf,
    params=['n_estimators', 'max_depth', 'min_samples_split']
)
fig.update_layout(
    title="Coordenadas Paralelas - Top Hiperparámetros"
)
fig.show()

print("\n🔗 Las líneas muestran combinaciones de hiperparámetros y su accuracy resultante")

---

## 🎓 Parte 5: Ejemplo Avanzado - Regresión con Pruning

El **pruning** es una característica poderosa de Optuna que detiene trials no prometedores temprano, ahorrando tiempo computacional.

### Ejemplo: Gradient Boosting Regressor en el Dataset de Diabetes

In [ ]:
# Cargar dataset de diabetes (regresión)
print("📊 Cargando dataset de Diabetes...")
diabetes = load_diabetes()
X_train_diab, X_test_diab, y_train_diab, y_test_diab = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42
)

print(f"✅ Datos preparados:")
print(f"   - Entrenamiento: {X_train_diab.shape[0]} muestras")
print(f"   - Test: {X_test_diab.shape[0]} muestras")
print(f"   - Características: {X_train_diab.shape[1]}")

In [ ]:
def objective_gradient_boosting_with_pruning(trial):
    """
    Función objetivo con pruning para Gradient Boosting.
    
    El pruning permite detener trials que no son prometedores
    sin completar todas las iteraciones.
    """
    
    # Hiperparámetros a optimizar
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }
    
    # Crear modelo
    model = GradientBoostingRegressor(**params, random_state=42)
    
    # Validación cruzada con seguimiento de iteraciones para pruning
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = []
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_diab)):
        X_fold_train = X_train_diab[train_idx]
        y_fold_train = y_train_diab[train_idx]
        X_fold_val = X_train_diab[val_idx]
        y_fold_val = y_train_diab[val_idx]
        
        # Entrenar
        model.fit(X_fold_train, y_fold_train)
        
        # Predecir y evaluar
        y_pred = model.predict(X_fold_val)
        mse = mean_squared_error(y_fold_val, y_pred)
        scores.append(mse)
        
        # Reportar valor intermedio para pruning
        # Optuna puede decidir detener este trial si va mal
        trial.report(mse, fold)
        
        # Verificar si debemos podar este trial
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    # Retornar MSE promedio (queremos minimizarlo)
    return np.mean(scores)

# Crear estudio con pruner
study_gb = optuna.create_study(
    direction='minimize',
    study_name='GradientBoosting_with_Pruning',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(  # Poda si está por debajo de la mediana
        n_startup_trials=5,  # No podar los primeros 5 trials
        n_warmup_steps=2     # Esperar 2 folds antes de empezar a podar
    )
)

# Optimizar
print("🔍 Optimizando Gradient Boosting con Pruning...")
print("✂️ Los trials no prometedores serán podados automáticamente\n")

study_gb.optimize(objective_gradient_boosting_with_pruning, n_trials=50, show_progress_bar=True)

# Resultados
print("\n" + "="*70)
print("📊 RESULTADOS - GRADIENT BOOSTING CON PRUNING")
print("="*70)
print(f"\n✅ Mejor MSE (CV): {study_gb.best_value:.2f}")
print(f"\n📌 Mejores hiperparámetros:")
for param, value in study_gb.best_params.items():
    print(f"   - {param}: {value}")

# Estadísticas de pruning
total_trials = len(study_gb.trials)
pruned_trials = len([t for t in study_gb.trials if t.state == optuna.trial.TrialState.PRUNED])
complete_trials = len([t for t in study_gb.trials if t.state == optuna.trial.TrialState.COMPLETE])

print(f"\n✂️ Estadísticas de Pruning:")
print(f"   - Trials completados: {complete_trials}")
print(f"   - Trials podados: {pruned_trials}")
print(f"   - Tasa de poda: {(pruned_trials/total_trials*100):.1f}%")
print(f"   - ⚡ Tiempo ahorrado estimado: ~{(pruned_trials/total_trials*100):.0f}%")

In [ ]:
# Evaluar mejor modelo en test
best_gb = GradientBoostingRegressor(**study_gb.best_params, random_state=42)
best_gb.fit(X_train_diab, y_train_diab)

y_pred_gb = best_gb.predict(X_test_diab)
test_mse = mean_squared_error(y_test_diab, y_pred_gb)
test_r2 = r2_score(y_test_diab, y_pred_gb)

print("\n" + "="*70)
print("🎯 EVALUACIÓN EN CONJUNTO DE TEST")
print("="*70)
print(f"\n📈 MSE en Test: {test_mse:.2f}")
print(f"📈 R² Score en Test: {test_r2:.4f}")

# Comparar con modelo por defecto
default_gb = GradientBoostingRegressor(random_state=42)
default_gb.fit(X_train_diab, y_train_diab)
default_mse = mean_squared_error(y_test_diab, default_gb.predict(X_test_diab))
default_r2 = r2_score(y_test_diab, default_gb.predict(X_test_diab))

print(f"\n🔄 Comparación:")
print(f"   MSE:")
print(f"      - Modelo optimizado: {test_mse:.2f}")
print(f"      - Modelo por defecto: {default_mse:.2f}")
print(f"      - Mejora: {((default_mse - test_mse) / default_mse * 100):.2f}%")
print(f"   R² Score:")
print(f"      - Modelo optimizado: {test_r2:.4f}")
print(f"      - Modelo por defecto: {default_r2:.4f}")

---

## 🎯 Parte 6: Diferentes Tipos de Hiperparámetros

Optuna soporta diferentes tipos de hiperparámetros. Veamos todos los tipos disponibles:

In [ ]:
def demo_tipos_hiperparametros(trial):
    """
    Demostración de todos los tipos de hiperparámetros
    que Optuna puede manejar.
    """
    
    # 1. ENTEROS (int)
    n_estimators = trial.suggest_int('n_estimators', 10, 100)
    # Con step (saltos)
    batch_size = trial.suggest_int('batch_size', 16, 128, step=16)  # 16, 32, 48, ...
    
    # 2. FLOTANTES (float)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1)
    # Con escala logarítmica (útil para learning rates)
    lr_log = trial.suggest_float('lr_log', 1e-5, 1e-1, log=True)
    
    # 3. CATEGÓRICOS (categorical)
    optimizer = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop'])
    activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'sigmoid'])
    
    # 4. DISCRETOS (discrete_uniform) - flotantes con paso
    dropout = trial.suggest_discrete_uniform('dropout', 0.1, 0.5, 0.1)  # 0.1, 0.2, 0.3, 0.4, 0.5
    
    # Imprimir ejemplos
    print(f"\n📊 Ejemplo de valores sugeridos:")
    print(f"   - n_estimators (int): {n_estimators}")
    print(f"   - batch_size (int con step): {batch_size}")
    print(f"   - learning_rate (float): {learning_rate:.6f}")
    print(f"   - lr_log (float logarítmico): {lr_log:.6f}")
    print(f"   - optimizer (categórico): {optimizer}")
    print(f"   - activation (categórico): {activation}")
    print(f"   - dropout (discreto): {dropout}")
    
    # Retornar un valor dummy para este ejemplo
    return np.random.random()

# Crear estudio de demostración
study_demo = optuna.create_study()
print("🎲 Ejecutando 3 trials de demostración de tipos de hiperparámetros:")
study_demo.optimize(demo_tipos_hiperparametros, n_trials=3)

---

## 🔧 Parte 7: Mejores Prácticas y Consejos

### 1. Controlar la Verbosidad del Output

In [ ]:
# Silenciar warnings de Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Crear callback personalizado para logging selectivo
def champion_callback(study, frozen_trial):
    """
    Solo imprime cuando se encuentra un nuevo mejor resultado.
    Útil para optimizaciones largas.
    """
    previous_best = study.user_attrs.get("previous_best_value", None)
    
    if previous_best is None or frozen_trial.value > previous_best:
        study.set_user_attr("previous_best_value", frozen_trial.value)
        improvement = ""
        if previous_best is not None:
            improvement = f" (+{((frozen_trial.value - previous_best) / previous_best * 100):.2f}%)"
        print(f"🏆 Trial {frozen_trial.number}: Nuevo mejor valor = {frozen_trial.value:.4f}{improvement}")

# Ejemplo de uso
study_quiet = optuna.create_study(direction='maximize')

def simple_objective(trial):
    x = trial.suggest_float('x', -10, 10)
    return -(x - 3)**2 + 10  # Máximo en x=3

print("🔇 Optimización con logging selectivo (solo mejoras):")
study_quiet.optimize(simple_objective, n_trials=30, callbacks=[champion_callback])

### 2. Persistencia de Estudios en Base de Datos

In [ ]:
# Guardar estudios en una base de datos SQLite
# Esto permite:
# - Reanudar optimizaciones interrumpidas
# - Compartir resultados
# - Ejecutar optimizaciones en paralelo

import sqlite3

# Crear base de datos
db_path = "optuna_studies.db"
storage = f"sqlite:///{db_path}"

# Crear estudio con almacenamiento persistente
study_persistent = optuna.create_study(
    study_name="mi_estudio_persistente",
    storage=storage,
    direction='maximize',
    load_if_exists=True  # Cargar si ya existe
)

def objective_persistente(trial):
    x = trial.suggest_float('x', -5, 5)
    return -(x**2)

# Primera ronda de optimización
print("💾 Guardando estudio en base de datos...")
study_persistent.optimize(objective_persistente, n_trials=20)
print(f"✅ Estudio guardado en: {db_path}")
print(f"📊 Trials completados: {len(study_persistent.trials)}")

# Simular cargar el estudio más tarde
print("\n🔄 Cargando estudio existente...")
study_loaded = optuna.load_study(
    study_name="mi_estudio_persistente",
    storage=storage
)
print(f"✅ Estudio cargado con {len(study_loaded.trials)} trials previos")

# Continuar optimización
print("\n▶️ Continuando optimización...")
study_loaded.optimize(objective_persistente, n_trials=10)
print(f"✅ Total de trials ahora: {len(study_loaded.trials)}")

### 3. Búsqueda Multi-Objetivo

Optuna también puede optimizar múltiples objetivos simultáneamente:

In [ ]:
def multi_objective_function(trial):
    """
    Optimización multi-objetivo: queremos maximizar accuracy Y minimizar tiempo.
    
    Ejemplo: encontrar un modelo que sea preciso pero también rápido.
    """
    # Simular hiperparámetros
    n_estimators = trial.suggest_int('n_estimators', 10, 100)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    
    # Objetivo 1: Accuracy (queremos maximizar)
    # Simulamos que más árboles = mejor accuracy
    accuracy = 0.7 + (n_estimators / 200) + (max_depth / 30) + np.random.random() * 0.1
    accuracy = min(accuracy, 1.0)
    
    # Objetivo 2: Tiempo de entrenamiento (queremos minimizar)
    # Simulamos que más árboles y profundidad = más tiempo
    training_time = n_estimators * 0.1 + max_depth * 0.5 + np.random.random() * 2
    
    # Retornar ambos objetivos
    return accuracy, training_time

# Crear estudio multi-objetivo
study_multi = optuna.create_study(
    directions=['maximize', 'minimize'],  # Maximizar accuracy, minimizar tiempo
    study_name='multi_objetivo'
)

print("🎯 Optimización Multi-Objetivo (Accuracy vs Tiempo)...")
study_multi.optimize(multi_objective_function, n_trials=50)

print("\n" + "="*70)
print("📊 RESULTADOS MULTI-OBJETIVO")
print("="*70)
print(f"\n🏆 Número de soluciones en Pareto Front: {len(study_multi.best_trials)}")

print("\n📋 Top 5 Soluciones (Trade-off Accuracy vs Tiempo):")
for i, trial in enumerate(study_multi.best_trials[:5], 1):
    print(f"\n   Solución {i}:")
    print(f"      - Accuracy: {trial.values[0]:.4f}")
    print(f"      - Tiempo: {trial.values[1]:.2f}s")
    print(f"      - Parámetros: {trial.params}")

In [ ]:
# Visualizar Pareto Front
fig = optuna.visualization.plot_pareto_front(
    study_multi, 
    target_names=['Accuracy', 'Tiempo (s)']
)
fig.update_layout(
    title="Frente de Pareto: Accuracy vs Tiempo de Entrenamiento"
)
fig.show()

print("\n💡 El Frente de Pareto muestra soluciones óptimas donde mejorar un objetivo empeora el otro")

---

## 📊 Parte 8: Análisis Completo de un Estudio

Veamos cómo extraer y analizar toda la información de un estudio:

In [ ]:
# Usar el estudio de Random Forest anterior
print("="*70)
print("🔍 ANÁLISIS COMPLETO DEL ESTUDIO - RANDOM FOREST")
print("="*70)

# 1. Información general
print(f"\n📌 Información General:")
print(f"   - Nombre del estudio: {study_rf.study_name}")
print(f"   - Dirección: {study_rf.direction}")
print(f"   - Total de trials: {len(study_rf.trials)}")

# 2. Mejor trial
print(f"\n🏆 Mejor Trial:")
print(f"   - Número: {study_rf.best_trial.number}")
print(f"   - Valor: {study_rf.best_value:.4f}")
print(f"   - Parámetros: {study_rf.best_params}")

# 3. Estadísticas de todos los trials
all_values = [t.value for t in study_rf.trials if t.value is not None]
print(f"\n📊 Estadísticas de Todos los Trials:")
print(f"   - Media: {np.mean(all_values):.4f}")
print(f"   - Mediana: {np.median(all_values):.4f}")
print(f"   - Std Dev: {np.std(all_values):.4f}")
print(f"   - Mínimo: {np.min(all_values):.4f}")
print(f"   - Máximo: {np.max(all_values):.4f}")

# 4. Top 5 mejores trials
print(f"\n🥇 Top 5 Mejores Trials:")
sorted_trials = sorted(study_rf.trials, key=lambda t: t.value if t.value else -np.inf, reverse=True)
for i, trial in enumerate(sorted_trials[:5], 1):
    print(f"\n   {i}. Trial #{trial.number}:")
    print(f"      - Accuracy: {trial.value:.4f}")
    print(f"      - n_estimators: {trial.params.get('n_estimators')}")
    print(f"      - max_depth: {trial.params.get('max_depth')}")

# 5. Crear DataFrame con todos los trials
trials_df = study_rf.trials_dataframe()
print(f"\n📋 DataFrame de Trials (primeras 5 columnas):")
print(trials_df[['number', 'value', 'state']].head())

# Guardar a CSV
trials_df.to_csv('optuna_trials_analysis.csv', index=False)
print(f"\n💾 Resultados guardados en: optuna_trials_analysis.csv")

---

## 🎨 Parte 9: Todas las Visualizaciones Disponibles

Resumen de todas las visualizaciones que Optuna ofrece:

In [ ]:
print("📊 RESUMEN DE VISUALIZACIONES DISPONIBLES EN OPTUNA\n")

visualizations = [
    {
        'nombre': 'Historial de Optimización',
        'función': 'plot_optimization_history(study)',
        'descripción': 'Muestra la evolución de los valores objetivo a lo largo de los trials'
    },
    {
        'nombre': 'Importancia de Parámetros',
        'función': 'plot_param_importances(study)',
        'descripción': 'Identifica qué hiperparámetros tienen mayor impacto'
    },
    {
        'nombre': 'Slice Plot',
        'función': 'plot_slice(study)',
        'descripción': 'Muestra la relación entre cada parámetro y el objetivo'
    },
    {
        'nombre': 'Coordenadas Paralelas',
        'función': 'plot_parallel_coordinate(study)',
        'descripción': 'Visualiza interacciones entre múltiples parámetros'
    },
    {
        'nombre': 'Gráfico de Contorno',
        'función': 'plot_contour(study)',
        'descripción': 'Relación 2D entre pares de parámetros'
    },
    {
        'nombre': 'EDF (Distribución Empírica)',
        'función': 'plot_edf(study)',
        'descripción': 'Distribución de valores objetivo alcanzados'
    },
    {
        'nombre': 'Historial de Hiperparámetros',
        'función': 'plot_param_history(study, param_name)',
        'descripción': 'Evolución de un hiperparámetro específico'
    },
    {
        'nombre': 'Frente de Pareto',
        'función': 'plot_pareto_front(study)',
        'descripción': 'Para optimización multi-objetivo'
    }
]

for i, viz in enumerate(visualizations, 1):
    print(f"{i}. {viz['nombre']}")
    print(f"   Función: {viz['función']}")
    print(f"   📝 {viz['descripción']}\n")

In [ ]:
# Generar algunas visualizaciones adicionales del estudio de RF

# Contour plot - relación entre pares de parámetros
fig = optuna.visualization.plot_contour(study_rf, params=['n_estimators', 'max_depth'])
fig.update_layout(title="Gráfico de Contorno: n_estimators vs max_depth")
fig.show()

In [ ]:
# EDF - Función de Distribución Empírica
fig = optuna.visualization.plot_edf(study_rf)
fig.update_layout(title="Función de Distribución Empírica de Accuracy")
fig.show()
print("\n💡 Muestra la proporción de trials que alcanzaron cada nivel de accuracy")

---

## 🎯 Parte 10: Ejemplo Final Completo - Pipeline Realista

Pongamos todo junto en un ejemplo realista completo con:
- Preprocesamiento de datos
- Optimización de hiperparámetros
- Validación cruzada
- Evaluación final
- Análisis de resultados

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Crear dataset sintético más complejo
print("🏗️ Creando dataset sintético complejo...")
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=3,
    n_clusters_per_class=2,
    random_state=42,
    class_sep=0.8
)

X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Dataset creado:")
print(f"   - Entrenamiento: {X_train_final.shape}")
print(f"   - Test: {X_test_final.shape}")
print(f"   - Características: {X_train_final.shape[1]}")
print(f"   - Balance de clases: {np.bincount(y_train_final)}")

In [ ]:
def objective_pipeline_completo(trial):
    """
    Función objetivo completa con pipeline de sklearn.
    """
    
    # 1. Elegir el algoritmo
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest'])
    
    # 2. Hiperparámetros específicos según el algoritmo
    if classifier_name == 'SVM':
        svc_c = trial.suggest_float('svc_c', 0.1, 100, log=True)
        svc_kernel = trial.suggest_categorical('svc_kernel', ['linear', 'rbf', 'poly'])
        
        if svc_kernel == 'rbf':
            svc_gamma = trial.suggest_float('svc_gamma', 1e-5, 1e-1, log=True)
            classifier = SVC(C=svc_c, kernel=svc_kernel, gamma=svc_gamma, random_state=42)
        else:
            classifier = SVC(C=svc_c, kernel=svc_kernel, random_state=42)
            
    else:  # RandomForest
        rf_n_estimators = trial.suggest_int('rf_n_estimators', 50, 300)
        rf_max_depth = trial.suggest_int('rf_max_depth', 3, 20)
        rf_min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 15)
        
        classifier = RandomForestClassifier(
            n_estimators=rf_n_estimators,
            max_depth=rf_max_depth,
            min_samples_split=rf_min_samples_split,
            random_state=42,
            n_jobs=-1
        )
    
    # 3. Crear pipeline con preprocesamiento
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', classifier)
    ])
    
    # 4. Validación cruzada
    scores = cross_val_score(
        pipeline, X_train_final, y_train_final, 
        cv=5, scoring='accuracy', n_jobs=-1
    )
    
    return scores.mean()

# Crear y ejecutar estudio
study_final = optuna.create_study(
    direction='maximize',
    study_name='Pipeline_Completo',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3)
)

print("\n🚀 Iniciando optimización de pipeline completo...")
print("⏳ Esto optimizará tanto el algoritmo como sus hiperparámetros\n")

study_final.optimize(objective_pipeline_completo, n_trials=100, show_progress_bar=True)

In [ ]:
# Análisis de resultados
print("\n" + "="*70)
print("🎯 RESULTADOS FINALES - PIPELINE COMPLETO")
print("="*70)

print(f"\n✅ Mejor Accuracy (CV): {study_final.best_value:.4f}")
print(f"\n🏆 Mejor Configuración:")
print(f"   - Algoritmo: {study_final.best_params['classifier']}")

# Mostrar parámetros específicos del algoritmo elegido
best_algo = study_final.best_params['classifier']
print(f"\n📌 Hiperparámetros del {best_algo}:")
for param, value in study_final.best_params.items():
    if param != 'classifier' and param.startswith(best_algo.lower()[:2]):
        print(f"   - {param}: {value}")

# Estadísticas por algoritmo
svm_trials = [t for t in study_final.trials if t.params.get('classifier') == 'SVM' and t.value]
rf_trials = [t for t in study_final.trials if t.params.get('classifier') == 'RandomForest' and t.value]

print(f"\n📊 Comparación de Algoritmos:")
if svm_trials:
    print(f"   SVM:")
    print(f"      - Trials: {len(svm_trials)}")
    print(f"      - Mejor: {max(t.value for t in svm_trials):.4f}")
    print(f"      - Media: {np.mean([t.value for t in svm_trials]):.4f}")

if rf_trials:
    print(f"   RandomForest:")
    print(f"      - Trials: {len(rf_trials)}")
    print(f"      - Mejor: {max(t.value for t in rf_trials):.4f}")
    print(f"      - Media: {np.mean([t.value for t in rf_trials]):.4f}")

In [ ]:
# Entrenar y evaluar el mejor modelo
print("\n" + "="*70)
print("🔬 EVALUACIÓN EN CONJUNTO DE TEST")
print("="*70)

# Reconstruir el mejor modelo
best_params = study_final.best_params.copy()
classifier_name = best_params.pop('classifier')
print("parametros del mejor modelo:", best_params)
if classifier_name == 'SVM':
    # Inicializar el diccionario de parámetros
    params = {}
    for k, v in best_params.items():
        if k.startswith('svc_'):
            # Si el parámetro es 'svc_c', renombrarlo a 'C'
            if k == 'svc_c':
                params['C'] = v
            else:
                # Para el resto de parámetros, quitar el prefijo 'svc_'
                params[k.replace('svc_', '')] = v
    # Mostrar los parámetros procesados (opcional, para depuración)
    print("Parámetros procesados para SVM:", params)
    best_classifier = SVC(**params, random_state=42)
else:  # Si el clasificador es RandomForest
    # Inicializar el diccionario de parámetros
    params = {}
    for k, v in best_params.items():
        if k.startswith('rf_'):
            # Quitar el prefijo 'rf_' a todos los parámetros
            params[k.replace('rf_', '')] = v
    # Mostrar los parámetros procesados (opcional, para depuración)
    print("Parámetros procesados para RandomForest:", params)
    best_classifier = RandomForestClassifier(**params, random_state=42, n_jobs=-1)


# Pipeline final
best_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', best_classifier)
])

# Entrenar con todos los datos de entrenamiento
best_pipeline.fit(X_train_final, y_train_final)

# Evaluar en test
y_pred_final = best_pipeline.predict(X_test_final)
test_accuracy_final = accuracy_score(y_test_final, y_pred_final)

print(f"\n📈 Resultados:")
print(f"   - Accuracy en CV (train): {study_final.best_value:.4f}")
print(f"   - Accuracy en Test: {test_accuracy_final:.4f}")
print(f"   - Diferencia: {abs(study_final.best_value - test_accuracy_final):.4f}")

if abs(study_final.best_value - test_accuracy_final) < 0.05:
    print(f"\n✅ El modelo generaliza bien (diferencia < 5%)")
else:
    print(f"\n⚠️ Posible sobreajuste (diferencia > 5%)")

# Matriz de confusión
from sklearn.metrics import confusion_matrix, classification_report

print(f"\n📊 Matriz de Confusión:")
cm = confusion_matrix(y_test_final, y_pred_final)
print(cm)

print(f"\n📋 Reporte de Clasificación:")
print(classification_report(y_test_final, y_pred_final, target_names=['Clase 0', 'Clase 1']))

In [ ]:
# Visualizaciones finales
print("\n📊 Generando visualizaciones finales...")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Historial de optimización
ax = axes[0, 0]
trials_numbers = [t.number for t in study_final.trials if t.value]
trials_values = [t.value for t in study_final.trials if t.value]
best_so_far = pd.Series(trials_values).cummax()

ax.plot(trials_numbers, trials_values, 'o', alpha=0.3, label='Cada trial')
ax.plot(trials_numbers, best_so_far, 'r-', linewidth=2, label='Mejor hasta ahora')
ax.set_xlabel('Número de Trial')
ax.set_ylabel('Accuracy (CV)')
ax.set_title('Convergencia de la Optimización')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Distribución de accuracy por algoritmo
ax = axes[0, 1]
svm_values = [t.value for t in svm_trials]
rf_values = [t.value for t in rf_trials]

positions = []
data_to_plot = []
labels = []

if svm_values:
    positions.append(1)
    data_to_plot.append(svm_values)
    labels.append(f'SVM\n(n={len(svm_values)})')

if rf_values:
    positions.append(2)
    data_to_plot.append(rf_values)
    labels.append(f'RF\n(n={len(rf_values)})')

bp = ax.boxplot(data_to_plot, positions=positions, labels=labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
ax.set_ylabel('Accuracy (CV)')
ax.set_title('Distribución de Accuracy por Algoritmo')
ax.grid(True, alpha=0.3)

# 3. Top parámetros (importancia)
ax = axes[1, 0]
# Calcular importancia manualmente (simplificado)
param_importance = {}
for param in study_final.best_params.keys():
    if param != 'classifier':
        # Correlación entre valor del parámetro y accuracy
        values = []
        accuracies = []
        for t in study_final.trials:
            if param in t.params and t.value:
                values.append(t.params[param] if isinstance(t.params[param], (int, float)) else 0)
                accuracies.append(t.value)
        if values and len(set(values)) > 1:
            param_importance[param] = abs(np.corrcoef(values, accuracies)[0, 1])

if param_importance:
    sorted_params = sorted(param_importance.items(), key=lambda x: x[1], reverse=True)[:5]
    params_names = [p[0] for p in sorted_params]
    params_values = [p[1] for p in sorted_params]
    
    ax.barh(params_names, params_values, color='steelblue')
    ax.set_xlabel('Importancia (correlación con accuracy)')
    ax.set_title('Top 5 Hiperparámetros Más Importantes')
    ax.grid(True, alpha=0.3, axis='x')

# 4. Matriz de confusión visual
ax = axes[1, 1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Clase 0', 'Clase 1'],
            yticklabels=['Clase 0', 'Clase 1'])
ax.set_ylabel('Verdadero')
ax.set_xlabel('Predicho')
ax.set_title('Matriz de Confusión (Test Set)')

plt.tight_layout()
plt.savefig('optuna_resultados_finales.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Visualizaciones guardadas en: optuna_resultados_finales.png")

---

## 📚 Parte 11: Resumen y Recursos Adicionales

### 🎓 Lo que hemos aprendido:

1. **Fundamentos de Optuna**
   - Qué es y por qué usarlo
   - Conceptos: Study, Trial, Sampler, Pruner

2. **Optimización Básica**
   - Funciones matemáticas simples
   - Diferentes tipos de hiperparámetros

3. **Machine Learning**
   - Clasificación con Random Forest
   - Regresión con Gradient Boosting
   - Pipelines completos con preprocesamiento

4. **Características Avanzadas**
   - Pruning automático
   - Optimización multi-objetivo
   - Persistencia en base de datos
   - Callbacks personalizados

5. **Visualización y Análisis**
   - 8+ tipos de gráficos diferentes
   - Análisis de importancia de hiperparámetros
   - Interpretación de resultados

---

### 🔑 Puntos Clave a Recordar:

✅ **Usa TPESampler** para optimización bayesiana eficiente (es el default)

✅ **Implementa pruning** para ahorrar tiempo en búsquedas largas

✅ **Usa validación cruzada** en la función objetivo para resultados más robustos

✅ **Guarda estudios en BD** para optimizaciones largas o distribuidas

✅ **Analiza visualizaciones** para entender qué hiperparámetros importan

✅ **Escala logarítmica** para learning rates y parámetros de regularización

✅ **Callbacks personalizados** para controlar el output en búsquedas largas

---

### 📖 Recursos Adicionales:

**Documentación Oficial:**
- [Optuna Documentation](https://optuna.readthedocs.io/)
- [Optuna Tutorial](https://optuna.readthedocs.io/en/stable/tutorial/index.html)
- [Optuna Examples](https://github.com/optuna/optuna-examples)

**Integraciones:**
- [Optuna + XGBoost](https://optuna.readthedocs.io/en/stable/reference/integration.html#xgboost)
- [Optuna + LightGBM](https://optuna.readthedocs.io/en/stable/reference/integration.html#lightgbm)
- [Optuna + PyTorch](https://optuna.readthedocs.io/en/stable/reference/integration.html#pytorch)
- [Optuna + TensorFlow/Keras](https://optuna.readthedocs.io/en/stable/reference/integration.html#tensorflow)

**Herramientas Visuales:**
- [Optuna Dashboard](https://github.com/optuna/optuna-dashboard) - Interfaz web interactiva

---

### 💡 Próximos Pasos Sugeridos:

1. **Practica con tus propios datasets**
   - Aplica lo aprendido a problemas reales
   - Experimenta con diferentes algoritmos

2. **Explora optimización distribuida**
   - Usa bases de datos para paralelizar búsquedas
   - Ejecuta en múltiples máquinas simultáneamente

3. **Integra con otras herramientas**
   - Combina con MLflow para tracking completo
   - Usa con frameworks como FastAPI para producción

4. **Profundiza en samplers avanzados**
   - CMA-ES para espacios continuos
   - NSGA-II para multi-objetivo

---

### 🎉 ¡Felicidades!

Has completado este tutorial completo de Optuna. Ahora tienes las herramientas para:

- ✅ Optimizar hiperparámetros eficientemente
- ✅ Ahorrar tiempo con pruning automático
- ✅ Analizar e interpretar resultados
- ✅ Implementar búsquedas en proyectos reales

**¡Sigue experimentando y optimizando! 🚀**

---

## 🔬 Apéndice: Comparación Práctica de Métodos de Búsqueda

Para finalizar, veamos una comparación directa entre diferentes estrategias de búsqueda:

In [ ]:
import time

# Función objetivo común
def benchmark_objective(trial):
    """Función objetivo para benchmark de métodos"""
    x = trial.suggest_float('x', -10, 10)
    y = trial.suggest_float('y', -10, 10)
    z = trial.suggest_float('z', -10, 10)
    
    # Función compleja con múltiples mínimos locales
    result = (
        np.sin(x) * np.cos(y) + 
        (x - 2)**2 + (y + 1)**2 + (z - 0.5)**2 +
        0.1 * np.sin(5 * x) * np.cos(5 * y)
    )
    return result

# Configurar diferentes samplers
samplers = {
    'TPE (Bayesiano)': optuna.samplers.TPESampler(seed=42),
    'Random Search': optuna.samplers.RandomSampler(seed=42),
    'Grid Search': optuna.samplers.GridSampler({
        'x': np.linspace(-10, 10, 10),
        'y': np.linspace(-10, 10, 10),
        'z': np.linspace(-10, 10, 10)
    })
}

results = {}

print("🏁 Benchmark de Métodos de Búsqueda")
print("="*70)

for name, sampler in samplers.items():
    print(f"\n▶️ Probando {name}...")
    
    study = optuna.create_study(
        direction='minimize',
        sampler=sampler
    )
    
    start_time = time.time()
    
    if name == 'Grid Search':
        # Grid search tiene número fijo de trials
        study.optimize(benchmark_objective, n_trials=1000)
    else:
        study.optimize(benchmark_objective, n_trials=100)
    
    elapsed_time = time.time() - start_time
    
    results[name] = {
        'best_value': study.best_value,
        'time': elapsed_time,
        'trials': len(study.trials)
    }
    
    print(f"   ✅ Mejor valor: {study.best_value:.6f}")
    print(f"   ⏱️ Tiempo: {elapsed_time:.2f}s")
    print(f"   🔢 Trials: {len(study.trials)}")

# Resumen comparativo
print("\n" + "="*70)
print("📊 RESUMEN COMPARATIVO")
print("="*70)

df_comparison = pd.DataFrame(results).T
df_comparison['trials_per_second'] = df_comparison['trials'] / df_comparison['time']

print("\n", df_comparison.to_string())

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Mejor valor encontrado
axes[0].bar(range(len(results)), [r['best_value'] for r in results.values()], 
            color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0].set_xticks(range(len(results)))
axes[0].set_xticklabels(results.keys(), rotation=15, ha='right')
axes[0].set_ylabel('Mejor Valor (menor es mejor)')
axes[0].set_title('Comparación de Resultados')
axes[0].grid(True, alpha=0.3, axis='y')

# 2. Tiempo de ejecución
axes[1].bar(range(len(results)), [r['time'] for r in results.values()],
            color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_xticks(range(len(results)))
axes[1].set_xticklabels(results.keys(), rotation=15, ha='right')
axes[1].set_ylabel('Tiempo (segundos)')
axes[1].set_title('Tiempo de Ejecución')
axes[1].grid(True, alpha=0.3, axis='y')

# 3. Eficiencia (trials por segundo)
axes[2].bar(range(len(results)), df_comparison['trials_per_second'],
            color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[2].set_xticks(range(len(results)))
axes[2].set_xticklabels(results.keys(), rotation=15, ha='right')
axes[2].set_ylabel('Trials por Segundo')
axes[2].set_title('Eficiencia')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('optuna_benchmark_metodos.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 Conclusión:")
print("   - TPE (Bayesiano) generalmente encuentra mejores resultados más rápido")
print("   - Random Search es simple pero puede ser ineficiente")
print("   - Grid Search es exhaustivo pero computacionalmente caro")
print("\n✅ Gráfico guardado en: optuna_benchmark_metodos.png")

---

## 🎯 Ejercicio Final para Practicar

**Reto:** Optimiza un clasificador SVM para el dataset de dígitos de sklearn con las siguientes condiciones:

1. Usa `load_digits()` de sklearn
2. Optimiza al menos 4 hiperparámetros
3. Implementa pruning
4. Usa validación cruzada de 5 folds
5. Genera al menos 3 visualizaciones
6. Compara con un modelo baseline (parámetros por defecto)

**Bonus:** Implementa optimización multi-objetivo (accuracy vs tiempo de predicción)

¡Buena suerte! 🚀

---

**Fin del Tutorial de Optuna** 🎓

¿Preguntas? Consulta la [documentación oficial](https://optuna.readthedocs.io/) o experimenta con el código de este notebook.

**Happy Optimizing! 🎉**